In [1]:
import os
import mne
from glob import glob
from mne.bem import make_watershed_bem, make_scalp_surfaces
from mne.coreg import Coregistration
from mne.io import read_info
from mne.report import Report
import pandas as pd
import json
import numpy as np

import pyvistaqt

from mne.viz import create_3d_figure

import pyvista

# Set PyVista to off-screen mode
#pyvista.OFF_SCREEN = True
#mne.viz.set_3d_backend('pyvista')
mne.viz.set_3d_backend('pyvistaqt') 

#os.environ['PYVISTA_OFF_SCREEN'] = 'true'
#os.environ['MNE_3D_BACKEND'] = 'pyvista'

Using pyvistaqt 3d backend.


In [3]:
#path to data
subjects_dir = '/scratch2/alinat/project/PD-EEG-ANON_eegOnly/derivatives/FS_SUBJECT_DIR' 
eeg_dir = '/scratch2/alinat/project/PD-EEG-ANON_eegOnly/derivatives/mne_pipeline_eye-closed_run1'


#load environments 
os.environ['FREESURFER_HOME'] = '/usr/local/freesurfer'
os.environ['SUBJECTS_DIR'] = subjects_dir 
os.environ['FS_LICENSE'] = '/usr/local/freesurfer/.license'

In [ ]:
#remove empty session folders in eeg directory

# Iterate over all 'sub-*' directories
for sub_dir in glob(os.path.join(eeg_dir, 'sub-*')):
    # Iterate over all 'ses-*' directories within each 'sub-*' directory
    for ses_dir in glob(os.path.join(sub_dir, 'ses-*')):
        eeg_folder = os.path.join(ses_dir, 'eeg')
        
        # Check if the 'ses-*' folder is empty or if the 'eeg/' subfolder is empty
        if not os.listdir(ses_dir) or (os.path.exists(eeg_folder) and not os.listdir(eeg_folder)):
            try:
                if os.path.exists(eeg_folder) and not os.listdir(eeg_folder):
                    os.rmdir(eeg_folder)  # Remove the empty 'eeg/' folder first
                os.rmdir(ses_dir)  # Remove the 'ses-*' folder
                print(f"Deleted empty session folder: {ses_dir}")
            except OSError as e:
                print(f"Could not delete {ses_dir}: {e}")


In [ ]:
#interpolate bad channels and make ad hoc and diagonal covariation matrix

# List of channels to exclude from the data
exclude_channels = ['M1', 'M2', 'VEO', 'F11', 'F12', 'FT11', 'FT12', 'FT7', 'FT8', 'PO5', 'PO6',
                    'm1', 'm2', 'veo', 'f11', 'f12', 'ft11', 'ft12', 'ft7', 'ft8', 'po5', 'po6']

# Loop over the paired raw and epoch files
for raw_file, epoch_file in zip(sorted(glob(eeg_dir + '/*/*/*/*clean_raw.fif')), 
                                sorted(glob(eeg_dir + '/*/*/*/*clean_epo.fif'))):
    # Process raw file
    raw = mne.io.read_raw_fif(raw_file, preload=True)
    # Drop the exclude channels completely from the raw data
    raw.drop_channels([ch for ch in exclude_channels if ch in raw.ch_names])
    #print(raw_file, raw.info['custom_ref_applied'])
    # Apply average reference and interpolate bad channels
    #raw.set_eeg_reference('average', projection=False)
    raw.interpolate_bads(reset_bads=True)
    raw, _ = mne.set_eeg_reference(raw, ref_channels="average", projection=True)
    
    # Save the processed raw file with a new suffix
    new_raw_filename = raw_file.replace('-clean_raw.fif', '-clean_badCh-interp_raw.fif')
    raw.save(new_raw_filename, overwrite=True)
    print(f"Processed and saved: {new_raw_filename}")
    
    # Compute ad hoc covariance matrix for the raw data
    raw_cov = mne.make_ad_hoc_cov(raw.info)
    # Create a new filename for the covariance matrix
    cov_filename = raw_file.replace('_proc-clean_raw.fif', '_ad-hoc_cov.fif')
    # Save the covariance matrix
    mne.write_cov(cov_filename, raw_cov, overwrite=True)
    print(f"Ad hoc covariance matrix calculated and saved: {cov_filename}")

    # Compute diagonal covariance matrix for the raw data
    noise_cov = mne.compute_raw_covariance(raw, method='empirical')
    # Set the covariance to be processed as diagonal
    noise_cov.as_diag()
    # Create a new filename for the covariance matrix
    noise_cov_filename = raw_file.replace('_proc-clean_raw.fif', '_diagonal_cov.fif')
    # Save the diagonal covariance matrix (optional)
    mne.write_cov(noise_cov_filename, noise_cov, overwrite=True)
    # Print the result to verify
    print(f"Diagonal noise covariance matrix calculated and saved: {noise_cov_filename}")
    
    # Process epoch file
    epochs = mne.read_epochs(epoch_file, preload=True)
    # Drop the exclude channels completely from the epochs data
    epochs.drop_channels([ch for ch in exclude_channels if ch in epochs.ch_names])
    # Interpolate bad channels only (average reference already applied)
    epochs.interpolate_bads(reset_bads=True)
    # Save the processed epoch file with a new suffix
    new_epoch_filename = epoch_file.replace('-clean_epo.fif', '-clean_badCh-interp_epo.fif')
    epochs.save(new_epoch_filename, overwrite=True)
    print(f"Processed and saved: {new_epoch_filename}")

In [ ]:
standard_subject = 'fsaverage1'

# Create the BEM surfaces for the fsaverage subject
mne.bem.make_watershed_bem(subject=standard_subject, subjects_dir=subjects_dir, overwrite=True)
make_scalp_surfaces(subject=standard_subject, subjects_dir=subjects_dir, overwrite=True)
# Visualize the BEM surfaces
mne.viz.plot_bem(subject=standard_subject, subjects_dir=subjects_dir, brain_surfaces='white', orientation='coronal')

In [ ]:
#fiducials for standard subject

standard_subject = 'fsaverage1'

mne.gui.coregistration(subject=standard_subject, subjects_dir=subjects_dir)

In [ ]:
# Make a coregistration (before fwd)

standard_subject = 'fsaverage1'

# Create a report object
report = Report(title='Coregistration with fsaverage')
# Loop through each EEG file to create and save coregistration plots
for file in sorted(glob(eeg_dir + '/*/*/*/*proc-clean_badCh-interp_raw.fif')): 
    subj_eeg = file.split('/')[-1].split('_')[0]
    ses_eeg = file.split('/')[-1].split('_')[1]

    raw_subj = mne.io.read_raw_fif(file, preload=False) 
    info = raw_subj.info

    #make coregistration and save
    coreg = Coregistration(info, standard_subject, subjects_dir, fiducials="auto")
    #coreg.fit_fiducials()
    coreg.set_fid_match('matched')
    coreg.set_grow_hair(2)
    coreg.fit_icp(n_iterations=100, 
                nasion_weight=10, 
                eeg_weight=5,
                hsp_weight=15,
                lpa_weight=10,
                rpa_weight=10,
                verbose=True)
    coreg.set_rotation([-0.1, 0.0, 0.0])

    #plot head coregistration
    fig = mne.viz.plot_alignment(info, trans=coreg.trans, 
                                subject=standard_subject,
                                subjects_dir=subjects_dir,
                                surfaces=['head-dense',  'inner_skull'],  # Add 'head' surface to show the scalp/face
                                show_axes=True,
                                dig=True,
                                mri_fiducials=True,
                                #interaction='3d',
                                )
    # Set the view to a specific camera angle before saving the figure
    # Options for 'view' can be: 'lateral', 'medial', 'rostral', 'caudal', 'dorsal', 'ventral', 'frontal', 'parietal'
    mne.viz.set_3d_view(fig, azimuth=45, elevation=90, distance=0.6, focalpoint=(0.0, 0.0, 0.0))  # Adjust azimuth and elevation for the desired angle

    # Save the 2D figure from the 3D view
    fig_image_path = f"{subjects_dir}/{standard_subject}/bem/{subj_eeg}_{ses_eeg}_coregistration.png"
    fig.plotter.screenshot(fig_image_path)
    report.add_image(fig_image_path, title=f"{subj_eeg}_{ses_eeg} Coregistration", 
                     caption=f"{subj_eeg}_{ses_eeg} coregistration with fsaverage")

    #save trans file
    mne.write_trans(f"{subjects_dir}/{standard_subject}/bem/{subj_eeg}_{ses_eeg}-trans.fif", coreg.trans, overwrite=True)

# Save the report as an HTML file in the `fsaverage/bem/` directory
report_filename = f"{subjects_dir}/{standard_subject}/bem/fsaverage_coregistration_report.html"
report.save(report_filename, overwrite=True)

In [ ]:
# Create forward solution for standard subject
# make forward solution itself
standard_subject = 'fsaverage1'

# Path for the existing report
report_filename = f"{subjects_dir}/{standard_subject}/bem/fsaverage_forward_report.html"

# Load existing report if it exists by reinitializing a new report (append content manually)
report = Report(title='Forward Solution with fsaverage')

# Set up source space
src = mne.setup_source_space(subject=standard_subject, spacing='oct6', subjects_dir=subjects_dir, add_dist=False)

# Save the source space to disk
src_filename = f"{subjects_dir}/{standard_subject}/bem/{standard_subject}-src.fif"
mne.write_source_spaces(src_filename, src, overwrite=True)


# Create BEM model
conductivity = [0.3, 0.006, 0.3]  # Typical for EEG: [scalp, skull, brain]
bem_model = mne.make_bem_model(subject=standard_subject, ico=4, conductivity=conductivity, subjects_dir=subjects_dir)
bem_solution = mne.make_bem_solution(bem_model)

# Save the BEM solution to disk
bem_solution_filename = f"{subjects_dir}/{standard_subject}/bem/{standard_subject}-bem-sol.fif"
mne.write_bem_solution(bem_solution_filename, bem_solution, overwrite=True)

# Loop through each individual trans file to create and save forward solutions
for trans_file in sorted(glob(f"{subjects_dir}/{standard_subject}/bem/*-trans.fif")):
    subj_eeg = trans_file.split('/')[-1].split('_')[0]
    ses_eeg = (('-').join(trans_file.split('/')[-1].split('_')[1].split('-')[:2]))

    # Load the corresponding EEG raw file to extract `info`
    eeg_file = f"{eeg_dir}/{subj_eeg}/{ses_eeg}/eeg/{subj_eeg}_{ses_eeg}_task-restEyesClosed_run-01_proc-clean_badCh-interp_raw.fif"
    if os.path.exists(eeg_file):
        raw_subj = mne.io.read_raw_fif(eeg_file, preload=False)
        info = raw_subj.info

        # Compute the forward solution
        fwd = mne.make_forward_solution(info, trans=trans_file, src=src, bem=bem_solution, meg=False, eeg=True)

        # Save the forward solution
        fwd_filename = f"{subjects_dir}/{standard_subject}/bem/{subj_eeg}_{ses_eeg}-fwd.fif"
        mne.write_forward_solution(fwd_filename, fwd, overwrite=True)

        # Compute and save inverse operators for both noise covariance matrices
        for cov_type in ["ad-hoc", "diagonal"]:
            cov_file = f"{eeg_dir}/{subj_eeg}/{ses_eeg}/eeg/{subj_eeg}_{ses_eeg}_task-restEyesClosed_run-01_{cov_type}_cov.fif"
            if os.path.exists(cov_file):
                print(f"Processing noise covariance file: {cov_file}")
                noise_cov = mne.read_cov(cov_file)

                # Convert diagonal matrix to full for both types
                if noise_cov.data.ndim == 1:
                    print(f"Converting {cov_type} covariance matrix to full matrix.")
                    # Get channel names from the covariance matrix
                    cov_channels = noise_cov.ch_names
                    # Check and align channels with `info`
                    if set(cov_channels) != set(info['ch_names']):
                        print(f"Channel mismatch detected for {cov_type}. Aligning channels...")
                        mne.channels.equalize_channels([info, noise_cov])
                        cov_channels = noise_cov.ch_names  # Update after alignment
                    # Convert diagonal to full matrix
                    diagonal_values = noise_cov.data
                    full_cov_matrix = np.zeros((len(diagonal_values), len(diagonal_values)))
                    np.fill_diagonal(full_cov_matrix, diagonal_values)
                    # Get bads and projs safely
                    try:
                        bads = noise_cov.bads  # Get bads from covariance if available
                    except AttributeError:
                        bads = info['bads']  # Default to info's bad channels
                    try:
                        projs = noise_cov.projs  # Get projs from covariance if available
                    except AttributeError:
                        projs = info['projs']  # Default to info's projectors
                    # Create a new Covariance object
                    noise_cov = mne.Covariance(
                        data=full_cov_matrix,
                        names=cov_channels,  # Use aligned channel names
                        bads=bads,           # Use bad channels
                        projs=projs,         # Use projectors
                        nfree=noise_cov.nfree  # Preserve degrees of freedom
                    )
                    print(f"{cov_type} covariance matrix converted to full matrix.")

                # Creating the inverse operator
                inverse_operator = mne.minimum_norm.make_inverse_operator(info, fwd, noise_cov)

                # Save the inverse operator
                inv_filename = f"{subjects_dir}/{standard_subject}/bem/{subj_eeg}_{ses_eeg}_{cov_type}_inv.fif"
                mne.minimum_norm.write_inverse_operator(inv_filename, inverse_operator, overwrite=True)
                print(f"Saved inverse operator with {cov_type} noise covariance for {subj_eeg}_{ses_eeg}")

        

        # Create a 3D figure silently (show=False prevents pop-up)
        fig = create_3d_figure(size=(600, 600), bgcolor='black', show=False)

        # Plot the forward solution alignment into the pre-created figure
        mne.viz.plot_alignment(
            info=info,
            trans=trans_file,
            subject=standard_subject,
            subjects_dir=subjects_dir,
            bem=bem_solution,
            src=fwd['src'],
            surfaces=['head', 'white'],
            eeg='original',
            dig=True,
            show_axes=True,
            mri_fiducials=True,
            fig=fig  # Use the pre-created figure
        )

        # Set the 3D view
        mne.viz.set_3d_view(fig, azimuth=45, elevation=90, distance=0.6, focalpoint=(0.0, 0.0, 0.0))

        # Save the 2D figure from the 3D view
        fig_image_path = f"{subjects_dir}/{standard_subject}/bem/{subj_eeg}_{ses_eeg}_forward.png"
        fig.plotter.screenshot(fig_image_path)
        report.add_image(fig_image_path, title=f"{subj_eeg}_{ses_eeg} Forward Solution",
                        caption=f"{subj_eeg}_{ses_eeg} forward solution with fsaverage", 
                        tags=f"{subj_eeg}_{ses_eeg}",
                        section='Forward Solutions')




        print(f"Processed forward solution for {subj_eeg}_{ses_eeg}")

# Save the updated report
report.save(report_filename, overwrite=True)


In [ ]:
#for individual brains

In [ ]:
#Create BEM surfaces and scalp surfaces for subjects

list_subj = sorted(os.listdir(subjects_dir))
list_subj.remove('fsaverage1')
list_subj.remove('fsaverage2')
list_subj.remove('fsaverage3')

for subject in list_subj:
    #create bem surfaces
    make_watershed_bem(subject=subject, subjects_dir=subjects_dir, overwrite=True)
    make_scalp_surfaces(subject=subject, subjects_dir=subjects_dir, overwrite=True)
    #plot
    fig = mne.viz.plot_bem(subject=subject, subjects_dir=subjects_dir, brain_surfaces='white', orientation='coronal')
    # Create a new report for each subject
    report = Report(title=f'BEM Surfaces Report for {subject}')
    report.add_figure(fig=fig, title=f'BEM for {subject}', section='BEM surfaces', replace=True)
    
    # Define the report file path in the subject's directory
    report_filename = os.path.join(subjects_dir, subject, 'bem', f'{subject}_bem_surfaces_report.html')
    
    # Save the report to the subject's folder
    report.save(report_filename, overwrite=True)

In [ ]:
#estimate fiducials manually
for subject in list_subj:
    mne.gui.coregistration(subject=subject, subjects_dir=subjects_dir)

In [ ]:
# Create forward solution for individual brains
list_subj = sorted(os.listdir(subjects_dir))
list_subj.remove('fsaverage')
list_subj.remove('fsaverage1')
list_subj.remove('fsaverage2')
list_subj.remove('fsaverage3')

# Loop through each subject
for subject in list_subj:

    sessions = [i.split('/')[-2] for i in sorted(glob(eeg_dir + '/' +subject+'/*/'+'eeg'))]

    for session in sessions:

        # Load EEG data for coregistration
        eeg_file = glob(f"{eeg_dir}/{subject}/{session}/eeg/{subject}_{session}_*_proc-clean_badCh-interp_raw.fif")[0]
        raw_subj = mne.io.read_raw_fif(eeg_file, preload=False)
        info = raw_subj.info

        # Make coregistration and save transformation
        coreg = Coregistration(info, subject, subjects_dir, fiducials="auto", on_defects='warn')
        coreg.set_fid_match('matched')
        coreg.set_grow_hair(2)
        coreg.fit_icp(n_iterations=100, nasion_weight=15, eeg_weight=5, hsp_weight=10, verbose=True)

        # Plot coregistration
        fig_coreg = mne.viz.plot_alignment(
            info, trans=coreg.trans, subject=subject, subjects_dir=subjects_dir,
            surfaces=['head-dense', 'outer_skull', 'inner_skull'], show_axes=True,
            dig=True, mri_fiducials=True
        )
        mne.viz.set_3d_view(fig_coreg, azimuth=45, elevation=90, distance=0.6, focalpoint=(0.0, 0.0, 0.0))

        # Save transformation file
        trans_filename = f"{subjects_dir}/{subject}/bem/{subject}_{session}-trans.fif"
        mne.write_trans(trans_filename, coreg.trans, overwrite=True)

        # Set up surface source space
        src = mne.setup_source_space(subject=subject, spacing='oct6', subjects_dir=subjects_dir, add_dist=False)

        # Save surface source space to disk
        src_filename = f"{subjects_dir}/{subject}/bem/{subject}_{session}-src.fif"
        mne.write_source_spaces(src_filename, src, overwrite=True)

        # Create BEM model and solution
        conductivity = [0.3, 0.006, 0.3]  # Typical for EEG: [scalp, skull, brain]
        bem_model = mne.make_bem_model(subject=subject, ico=4, conductivity=conductivity, subjects_dir=subjects_dir)
        bem_solution = mne.make_bem_solution(bem_model)

        # Save BEM solution to disk
        bem_solution_filename = f"{subjects_dir}/{subject}/bem/{subject}_{session}-bem-sol.fif"
        mne.write_bem_solution(bem_solution_filename, bem_solution, overwrite=True)

        # Compute the forward solution
        fwd = mne.make_forward_solution(info, trans=coreg.trans, src=src, bem=bem_solution, meg=False, eeg=True)
        fwd_filename = f"{subjects_dir}/{subject}/bem/{subject}_{session}-fwd.fif"
        mne.write_forward_solution(fwd_filename, fwd, overwrite=True)

        # Compute and save inverse operators for both noise covariance matrices
        for cov_type in ["ad-hoc", "diagonal"]:
            cov_file = glob(f"{eeg_dir}/{subject}/{session}/eeg/{subject}_{session}_*_{cov_type}_cov.fif")[0]
            if os.path.exists(cov_file):
                print(f"Processing noise covariance file: {cov_file}")
                noise_cov = mne.read_cov(cov_file)

                # Convert diagonal matrix to full if needed
                if noise_cov.data.ndim == 1:
                    print(f"Converting {cov_type} covariance matrix to full matrix.")
                                    # Get channel names from the covariance matrix
                    cov_channels = noise_cov.ch_names
                                    # Check and align channels with `info`
                    if set(cov_channels) != set(info['ch_names']):
                        print(f"Channel mismatch detected for {cov_type}. Aligning channels...")
                        mne.channels.equalize_channels([info, noise_cov])
                        cov_channels = noise_cov.ch_names  # Update after alignment
                    # Convert diagonal to full matrix
                    diagonal_values = noise_cov.data
                    full_cov_matrix = np.zeros((len(diagonal_values), len(diagonal_values)))
                    np.fill_diagonal(full_cov_matrix, diagonal_values)
                    # Get bads and projs safely
                    try:
                        bads = noise_cov.bads  # Get bads from covariance if available
                    except AttributeError:
                        bads = info['bads']  # Default to info's bad channels
                    try:
                        projs = noise_cov.projs  # Get projs from covariance if available
                    except AttributeError:
                        projs = info['projs']  # Default to info's projectors
                    # Create a new Covariance object
                    noise_cov = mne.Covariance(
                        data=full_cov_matrix,
                        names=cov_channels,  # Use aligned channel names
                        bads=bads,           # Use bad channels
                        projs=projs,         # Use projectors
                        nfree=noise_cov.nfree  # Preserve degrees of freedom
                    )
                    print(f"{cov_type} covariance matrix converted to full matrix.")

                # Create the inverse operator
                inverse_operator = mne.minimum_norm.make_inverse_operator(info, fwd, noise_cov)

                # Save the inverse operator
                inv_filename = f"{subjects_dir}/{subject}/bem/{subject}_{session}_{cov_type}_inv.fif"
                mne.minimum_norm.write_inverse_operator(inv_filename, inverse_operator, overwrite=True)
                print(f"Saved inverse operator with {cov_type} noise covariance for {subject}")


        # Plot forward solution alignment
        fig_fwd = mne.viz.plot_alignment(
            info=info, trans=coreg.trans, subject=subject, subjects_dir=subjects_dir,
            bem=bem_solution, src=fwd['src'], surfaces=['head', 'white'], eeg='original',
            dig=True, show_axes=True, mri_fiducials=True
        )
        mne.viz.set_3d_view(fig_fwd, azimuth=45, elevation=90, distance=0.6, focalpoint=(0.0, 0.0, 0.0))

        # Create a report for each subject
        report = mne.Report(title=f"{subject}_{session} Coregistration and Forward Solution")
        report.add_figure(fig=fig_coreg, title="Coregistration", caption=f"{subject}_{session} Coregistration", tags='coregistration')
        report.add_figure(fig=fig_fwd, title="Forward Solution", caption=f"{subject}_{session} Forward Solution", tags='forward_solution')

        # Save the report
        report_filename = f"{subjects_dir}/{subject}/bem/{subject}_{session}_report.html"
        report.save(report_filename, overwrite=True)

        print(f"Report created and saved for {subject}: {report_filename}")


In [ ]:
# import os
# import numpy as np
# import mne
# from glob import glob
# from mne.coreg import Coregistration

# # Lists to store subject IDs for logging
# skipped_subjects = []  # Subjects where both BEM models failed
# one_layer_subjects = []  # Subjects that used a one-layer BEM

# # Create forward solution for individual brains
# list_subj = sorted(os.listdir(subjects_dir))
# list_subj.remove('fsaverage')
# list_subj.remove('fsaverage1')
# list_subj.remove('fsaverage2')
# list_subj.remove('fsaverage3')

# # Loop through each subject
# for subject in list_subj[150:]:
#     sessions = [i.split('/')[-2] for i in sorted(glob(eeg_dir + '/' +subject+'/*/'+'eeg'))]

#     for session in sessions:
#         try:
#             # Load EEG data for coregistration
#             eeg_file = glob(f"{eeg_dir}/{subject}/{session}/eeg/{subject}_{session}_*_proc-clean_badCh-interp_raw.fif")[0]
#             raw_subj = mne.io.read_raw_fif(eeg_file, preload=False)
#             info = raw_subj.info

#             # Make coregistration and save transformation
#             coreg = Coregistration(info, subject, subjects_dir, fiducials="auto", on_defects='warn')
#             coreg.set_fid_match('matched')
#             coreg.set_grow_hair(2)
#             coreg.fit_icp(n_iterations=100, nasion_weight=15, eeg_weight=5, hsp_weight=10, verbose=True)

#             # Save transformation file
#             trans_filename = f"{subjects_dir}/{subject}/bem/{subject}_{session}-trans.fif"
#             mne.write_trans(trans_filename, coreg.trans, overwrite=True)

#             # Set up surface source space
#             src = mne.setup_source_space(subject=subject, spacing='oct6', subjects_dir=subjects_dir, add_dist=False)

#             # Save surface source space to disk
#             src_filename = f"{subjects_dir}/{subject}/bem/{subject}_{session}-src.fif"
#             mne.write_source_spaces(src_filename, src, overwrite=True)

#             # Try three-layer BEM first
#             try:
#                 conductivity = [0.3, 0.006, 0.3]  # Typical for EEG: [scalp, skull, brain]
#                 bem_model = mne.make_bem_model(subject=subject, ico=4, conductivity=conductivity, subjects_dir=subjects_dir)
#             except RuntimeError:
#                 print(f"⚠️ Three-layer BEM failed for {subject}, trying one-layer BEM...")
#                 try:
#                     bem_model = mne.make_bem_model(subject=subject, ico=4, conductivity=[0.3], subjects_dir=subjects_dir)
#                     one_layer_subjects.append(subject)
#                 except RuntimeError:
#                     print(f"❌ One-layer BEM also failed for {subject}, skipping...")
#                     skipped_subjects.append(subject)
#                     continue  # Skip this subject
            
#             # Compute BEM solution
#             bem_solution = mne.make_bem_solution(bem_model)

#             # Save BEM solution to disk
#             bem_solution_filename = f"{subjects_dir}/{subject}/bem/{subject}_{session}-bem-sol.fif"
#             mne.write_bem_solution(bem_solution_filename, bem_solution, overwrite=True)

#             # Compute forward solution
#             fwd = mne.make_forward_solution(info, trans=coreg.trans, src=src, bem=bem_solution, meg=False, eeg=True)
#             fwd_filename = f"{subjects_dir}/{subject}/bem/{subject}_{session}-fwd.fif"
#             mne.write_forward_solution(fwd_filename, fwd, overwrite=True)

#             # Compute and save inverse operators for both noise covariance matrices
#             for cov_type in ["ad-hoc", "diagonal"]:
#                 cov_file = glob(f"{eeg_dir}/{subject}/{session}/eeg/{subject}_{session}_*_{cov_type}_cov.fif")[0]
#                 if os.path.exists(cov_file):
#                     print(f"Processing noise covariance file: {cov_file}")
#                     noise_cov = mne.read_cov(cov_file)
#                     inverse_operator = mne.minimum_norm.make_inverse_operator(info, fwd, noise_cov)
#                     inv_filename = f"{subjects_dir}/{subject}/bem/{subject}_{session}_{cov_type}_inv.fif"
#                     mne.minimum_norm.write_inverse_operator(inv_filename, inverse_operator, overwrite=True)
#                     print(f"Saved inverse operator with {cov_type} noise covariance for {subject}")

#             # Plot coregistration
#             fig_coreg = mne.viz.plot_alignment(
#                 info, trans=coreg.trans, subject=subject, subjects_dir=subjects_dir,
#                 surfaces=['head-dense', 'outer_skull', 'inner_skull'], show_axes=True,
#                 dig=True, mri_fiducials=True
#             )
#             mne.viz.set_3d_view(fig_coreg, azimuth=45, elevation=90, distance=0.6, focalpoint=(0.0, 0.0, 0.0))

#             # Plot forward solution alignment
#             fig_fwd = mne.viz.plot_alignment(
#                 info=info, trans=coreg.trans, subject=subject, subjects_dir=subjects_dir,
#                 bem=bem_solution, src=fwd['src'], surfaces=['head', 'white'], eeg='original',
#                 dig=True, show_axes=True, mri_fiducials=True
#             )
#             mne.viz.set_3d_view(fig_fwd, azimuth=45, elevation=90, distance=0.6, focalpoint=(0.0, 0.0, 0.0))

#             # Create a report for each subject
#             report = mne.Report(title=f"{subject}_{session} Coregistration and Forward Solution")
#             report.add_figure(fig=fig_coreg, title="Coregistration", caption=f"{subject}_{session} Coregistration", tags='coregistration')
#             report.add_figure(fig=fig_fwd, title="Forward Solution", caption=f"{subject}_{session} Forward Solution", tags='forward_solution')

#             # Save the report
#             report_filename = f"{subjects_dir}/{subject}/bem/{subject}_{session}_report.html"
#             report.save(report_filename, overwrite=True)

#             print(f"Report created and saved for {subject}: {report_filename}")

#         except Exception as e:
#             print(f"⚠️ Unexpected error for {subject}: {e}. Skipping...")
#             skipped_subjects.append(subject)
#             continue

# # Print results at the end
# print("\n✅ Processing Complete!")
# print(f"Subjects that used one-layer BEM: {one_layer_subjects}")
# print(f"Subjects that were skipped: {skipped_subjects}")
